In [43]:
# 📌 Import necessary libraries for data handling, visualization, and modeling
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import sklearn.model_selection

#import classifier algorithm here
from sklearn.linear_model import LogisticRegression

#import preprocessing module
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

# import evaluation metrics
from sklearn.metrics import confusion_matrix, accuracy_score

# Suppress warnings for better readability
import warnings
warnings.filterwarnings('ignore')

# print("Libraries imported successfully!")

# 📥 Load datasets
train = pd.read_csv('Train.csv')  # Training dataset
test = pd.read_csv('Test.csv')  # Test dataset (no labels)
ss = pd.read_csv('SampleSubmission.csv')  # Sample submission format
variables = pd.read_csv('VariableDefinitions.csv')  # Data dictionary

# Display first few rows to understand structure
print("🔹 Training Data Preview:")
display(train.head())

print("🔹 Variable Definitions Preview:")
display(variables.head())

# 📊 Check dataset shapes
# print(f"✅ Train dataset: {train.shape[0]} rows, {train.shape[1]} columns")
# print(f"✅ Test dataset: {test.shape[0]} rows, {test.shape[1]} columns")

# Check for missing values
# print('missing values:', train.isnull().sum())

# Explore Target distribution
# sns.catplot(x="bank_account", kind="count", data=train)

# Convert target label to numerical Data
le = LabelEncoder()
train['bank_account'] = le.fit_transform(train['bank_account'])

#Separate training features from target
X_train = train.drop(['bank_account'], axis=1)
y_train = train['bank_account']
print(y_train)

# function to preprocess our data from train models
def preprocessing_data(data):

    # Convert the following numerical labels from interger to float
    float_array = data[["household_size", "age_of_respondent", "year"]].values.astype(float)

    # categorical features to be onverted to One Hot Encoding
    categ = ["relationship_with_head",
             "marital_status",
             "education_level",
             "job_type",
             "country"]

    # One Hot Encoding conversion
    data = pd.get_dummies(data, prefix_sep="_", columns=categ)

    # Label Encoder conversion
    data["location_type"] = le.fit_transform(data["location_type"])
    data["cellphone_access"] = le.fit_transform(data["cellphone_access"])
    data["gender_of_respondent"] = le.fit_transform(data["gender_of_respondent"])

    # drop uniquid column
    data = data.drop(["uniqueid"], axis=1)

    # scale our data into range of 0 and 1
    scaler = MinMaxScaler(feature_range=(0, 1))
    data = scaler.fit_transform(data)

    return data

# preprocess the train data
processed_train = preprocessing_data(X_train)
processed_test = preprocessing_data(test)

X_Train, X_Val, y_Train, y_val = train_test_split(processed_train, y_train, stratify = y_train, test_size = 0.1, random_state=42)

clf = LogisticRegression(max_iter=5000, class_weight="balanced")
clf.fit(X_Train, y_Train)

clf_model = clf.predict(X_Val)
print("Error rate of LogisticRegression classifier: ", 1 - accuracy_score(y_val, clf_model))

test.bank_account = clf.predict(processed_test)

submission = pd.DataFrame({"uniqueid": test["uniqueid"] + " x " + test["country"], "bank_account": test.bank_account})
submission.sample(5)

submission.to_csv('first_submission.csv', index = False)

🔹 Training Data Preview:


,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3,24,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5,70,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5,26,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5,34,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8,26,Male,Child,Single/Never Married,Primary education,Informally employed


🔹 Variable Definitions Preview:


,Variable Definitions,Unnamed: 1
0,country,Country interviewee is in.
1,year,Year survey was done in.
2,uniqueid,Unique identifier for each interviewee
3,location_type,"Type of location: Rural, Urban"
4,cellphone_access,"If interviewee has access to a cellphone: Yes, No"


0        1
1        0
2        1
3        0
4        0
        ..
23519    0
23520    0
23521    0
23522    0
23523    0
Name: bank_account, Length: 23524, dtype: int64
Error rate of LogisticRegression classifier:  0.2061198470038249
